In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Date & Temporal Features
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day_of_Week'] = df['Date'].dt.dayofweek

# 2. Categorical Feature Encoding
encoders = {}
cat_cols = ['District', 'Mandi_Name', 'Crop', 'Variety']
for col in cat_cols:
    le = LabelEncoder()
    df[f'{col}_Encoded'] = le.fit_transform(df[col])
    encoders[col] = le

# 3. Feature Engineering: Distance, Transport Cost and Net Expected Price
# We will simulate a baseline distance (in km) to each mandi for a hypothetical farmer
# by mapping each Mandi_Name to a fixed distance.
np.random.seed(42)
unique_mandis = df['Mandi_Name'].unique()
mandi_distance_map = {mandi: np.random.uniform(5, 50) for mandi in unique_mandis}
df['Distance_KM'] = df['Mandi_Name'].map(mandi_distance_map)

# Transport cost per Quintal (1 Ton = 10 Quintals)
# Assuming a rate of 2 INR per KM per Quintal
TRANSPORT_RATE_PER_KM_QUINTAL = 2.0
df['Transport_Cost_Per_Quintal'] = df['Distance_KM'] * TRANSPORT_RATE_PER_KM_QUINTAL

# Modal_Price is usually given per Quintal in Indian mandis.
df['Net_Expected_Price'] = df['Modal_Price'] - df['Transport_Cost_Per_Quintal']

# 4. Feature Engineering: Rolling Averages (Price Volatility / Trends)
# Sort dataframe to perform logical rolling time-series calculations per Crop and Mandi
df = df.sort_values(by=['Crop', 'Mandi_Name', 'Date']).reset_index(drop=True)

df['Rolling_Mean_7D'] = df.groupby(['Crop', 'Mandi_Name'])['Modal_Price'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df['Rolling_Mean_30D'] = df.groupby(['Crop', 'Mandi_Name'])['Modal_Price'].transform(lambda x: x.rolling(30, min_periods=1).mean())

# 5. Feature Scaling
scaler = StandardScaler()
scale_cols = ['Arrival_Quantity_Tons', 'Min_Price', 'Max_Price', 'Modal_Price', 'Net_Expected_Price']
df[[f'{col}_Scaled' for col in scale_cols]] = scaler.fit_transform(df[scale_cols])

display(df.head())


,Date,State,District,Mandi_Name,Crop,Variety,Arrival_Quantity_Tons,Min_Price,Max_Price,Modal_Price,...,Distance_KM,Transport_Cost_Per_Quintal,Net_Expected_Price,Rolling_Mean_7D,Rolling_Mean_30D,Arrival_Quantity_Tons_Scaled,Min_Price_Scaled,Max_Price_Scaled,Modal_Price_Scaled,Net_Expected_Price_Scaled
0,2023-01-15,Telangana,Yadadri Bhuvanagiri,Aler APMC,Chana,FAQ (Fair Average Quality),159.0,4409,5374,4933,...,13.253203,26.506406,4906.493594,4933.000000,4933.000000,-0.123910,-0.365045,-0.341911,-0.339132,-0.332160
1,2023-04-09,Telangana,Yadadri Bhuvanagiri,Aler APMC,Chana,FAQ (Fair Average Quality),193.4,4527,5346,4960,...,13.253203,26.506406,4933.493594,4946.500000,4946.500000,0.264241,-0.323346,-0.350137,-0.330464,-0.323492
2,2023-04-30,Telangana,Yadadri Bhuvanagiri,Aler APMC,Chana,Hybrid,81.4,4969,6059,5495,...,13.253203,26.506406,5468.493594,5129.333333,5129.333333,-0.999505,-0.167151,-0.140653,-0.158695,-0.151742
3,2023-05-14,Telangana,Yadadri Bhuvanagiri,Aler APMC,Chana,FAQ (Fair Average Quality),152.6,4898,5648,5238,...,13.253203,26.506406,5211.493594,5156.500000,5156.500000,-0.196124,-0.192241,-0.261408,-0.241208,-0.234246
4,2024-04-21,Telangana,Yadadri Bhuvanagiri,Aler APMC,Chana,Grade A,285.1,5473,6941,6217,...,13.253203,26.506406,6190.493594,5368.600000,5368.600000,1.298934,0.010954,0.118483,0.073113,0.080039
